# Gold: North Star Metric (`governor_nsm`)

ADR [0022](../../docs/adr/0022-notebooks-de-diagnostico-medallion-separados-da-narrativa-do-tcc.md)
(issue [#128](https://github.com/Vini0606/Tecnicas-de-Ciencia-de-Dados-em-dados-do-Instagram/issues/128)).
Diagnóstico de `governor_nsm` -- `(comentários positivos / comentários totais) x alcance médio`
(`src/modeling/nsm_scorer.py` / `NsmScorer`, ADR 0020 Ficha 5). No produto final (`CONTEXT.md`) esta
métrica aparece como **"Engajamento qualificado"**, nunca como a sigla NSM. `alcance_medio` é um
**proxy** (`TOTAL ENGAJAMENTO / count` de `governor_engagement`), não alcance/views real -- o
Instagram não expõe alcance para posts estáticos.

Notebook estritamente leitura via `DeltaRepository`, mesmo princípio da ADR
[0003](../../docs/adr/0003-desacoplar-modelagem-do-notebook-via-scripts-cli-com-checkpoint.md).

In [ ]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

from config import settings
from src.repositories.delta_repository import DeltaRepository
from src.visualization.charts import plot_correlation_heatmap
from src.analysis.medallion_diagnostics import (
    completeness_summary,
    count_duplicate_rows,
    with_governor_metadata,
)

load_dotenv()
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)

repo = DeltaRepository(gold_dir=settings.GOLD_DIR, silver_dir=settings.SILVER_DIR)

## 1. Carga + schema

`inputUrl`/`username` são `nullable=True` por contrato de schema -- o merge `outer` de
`NsmScorer.score` pode gerar uma linha só de um dos dois lados (comentário sem par em
`governor_engagement` naquela execução, ou vice-versa).

In [ ]:
df_nsm = repo.load_nsm()
df_nsm.dtypes

## 2. Completude

In [ ]:
completude = completeness_summary(df_nsm)
print(f"linhas duplicadas (por inputUrl): {count_duplicate_rows(df_nsm, subset=['inputUrl'])}")
completude[completude['n_nulos'] > 0]

## 3. Distribuições univariadas

In [ ]:
metricas_chave = ['nsm', 'proporcao_positivos', 'alcance_medio', 'n_comentarios_totais']
fig, axes = plt.subplots(1, len(metricas_chave), figsize=(4 * len(metricas_chave), 4))
for ax, coluna in zip(axes, metricas_chave):
    sns.histplot(df_nsm[coluna], kde=True, ax=ax)
    ax.set_title(coluna)
plt.tight_layout()
plt.show()

## 4. Evolução temporal

Sem tabela `_history` irmã para `governor_nsm` hoje -- seção do esqueleto padrão pulada de
propósito.

## 5. Relação com covariáveis (partido/UF) e entre componentes

In [ ]:
df_com_metadado = with_governor_metadata(df_nsm, repo.load_governors_metadata())

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df_com_metadado, x='partido', y='nsm', ax=ax)
ax.set_title('NSM (Engajamento qualificado) por partido')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
plot_correlation_heatmap(df_nsm[metricas_chave])

## 6. Outliers

NSM extremo é esperado ser instável quando `n_comentarios_totais` é baixo -- sinalizamos os dois
juntos (extremo em NSM E poucos comentários), não só o extremo isolado.

In [ ]:
mediana_comentarios = df_nsm['n_comentarios_totais'].median()
media_nsm = df_nsm['nsm'].mean()
desvio_nsm = df_nsm['nsm'].std()

extremos = df_nsm[(df_nsm['nsm'] - media_nsm).abs() > 1.5 * desvio_nsm]
extremos_instaveis = extremos[extremos['n_comentarios_totais'] < mediana_comentarios]

print(f'{len(extremos)} perfis com NSM extremo (>1.5 desvio-padrão); {len(extremos_instaveis)} deles com poucos comentários (< mediana={mediana_comentarios:.0f})')
extremos[['username', 'nsm', 'n_comentarios_totais', 'proporcao_positivos', 'alcance_medio']].sort_values('nsm', ascending=False)

## Nota de interpretação

Um NSM alto sustentado por poucos comentários totais é um extremo estatístico, não
necessariamente um sinal de "engajamento qualificado" real -- checar `n_comentarios_totais` antes
de qualquer ranking editorial baseado nesta tabela.